# Tuần 2 – Nhóm 3: Tìm kiếm hình ảnh sản phẩm tương đồng
**Dataset:** Shopee – Price Match Guarantee (Kaggle 2021)  
**Mục tiêu:** Đọc dữ liệu, làm sạch, phân tích ban đầu, xây dựng baseline

## 0. Cài đặt thư viện

In [ ]:
# Chạy cell này một lần để cài thư viện
# !pip install pandas numpy matplotlib seaborn Pillow torch torchvision scikit-learn faiss-cpu tqdm

## 1. Import thư viện

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm import tqdm

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print('Import thư viện thành công!')

## 2. Cấu hình đường dẫn
> **Sửa `DATA_DIR` cho đúng với máy bạn trước khi chạy**

In [ ]:
# ============================================================
# THAY ĐỔI ĐƯỜNG DẪN NÀY CHO ĐÚNG VỚI MÁY BẠN
DATA_DIR = Path(r'd:\Study\docs\Python\shopee-product-matching')
# ============================================================

TRAIN_CSV    = DATA_DIR / 'train.csv'
TEST_CSV     = DATA_DIR / 'test.csv'
TRAIN_IMGDIR = DATA_DIR / 'train_images'
TEST_IMGDIR  = DATA_DIR / 'test_images'

# Kiểm tra tồn tại
for p in [TRAIN_CSV, TEST_CSV, TRAIN_IMGDIR, TEST_IMGDIR]:
    status = '✅' if p.exists() else '❌ KHÔNG TÌM THẤY'
    print(f'{status}  {p}')

: 

## 3. Đọc dữ liệu CSV

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train: {df_train.shape[0]:,} dòng × {df_train.shape[1]} cột')
print(f'Test : {df_test.shape[0]:,} dòng × {df_test.shape[1]} cột')
df_train.head()

In [ ]:
# Kiểu dữ liệu từng cột
df_train.info()

### 3.1 Bảng mô tả bộ dữ liệu

In [ ]:
n_groups = df_train['label_group'].nunique()
group_sizes = df_train.groupby('label_group').size()

summary = {
    'Tên bộ dữ liệu'        : 'Shopee – Price Match Guarantee',
    'Nguồn'                  : 'Kaggle 2021 (shopee-product-matching)',
    'Số mẫu (train)'         : f"{df_train.shape[0]:,}",
    'Số cột'                 : df_train.shape[1],
    'Cột nhãn'               : 'label_group',
    'Số nhóm sản phẩm'       : f"{n_groups:,}",
    'Trung bình ảnh/nhóm'    : f"{group_sizes.mean():.2f}",
    'Nhóm lớn nhất'          : int(group_sizes.max()),
    'Nhóm nhỏ nhất'          : int(group_sizes.min()),
    'Loại bài toán'          : 'Truy xuất / gợi ý hình ảnh (Image Retrieval)',
}

for k, v in summary.items():
    print(f'{k:<30}: {v}')

## 4. Làm sạch dữ liệu

In [ ]:
# 4.1 Kiểm tra giá trị thiếu
print('=== GIÁ TRỊ THIẾU ===')
print(df_train.isnull().sum())

# 4.2 Kiểm tra dòng trùng lặp
dup = df_train.duplicated().sum()
print(f'\nSố dòng trùng lặp: {dup}')

# 4.3 Kiểm tra posting_id trùng
dup_id = df_train['posting_id'].duplicated().sum()
print(f'Số posting_id trùng: {dup_id}')

In [ ]:
# 4.4 Kiểm tra file ảnh có tồn tại không (lấy mẫu 500 ảnh để nhanh)
sample_check = df_train.sample(min(500, len(df_train)), random_state=42)
missing_imgs = []

for fname in tqdm(sample_check['image'], desc='Kiểm tra ảnh'):
    path = TRAIN_IMGDIR / fname
    if not path.exists():
        missing_imgs.append(fname)

print(f'Ảnh thiếu (trong 500 mẫu): {len(missing_imgs)}')
if missing_imgs:
    print(missing_imgs[:5])

In [ ]:
# 4.5 Kiểm tra ảnh có load được không và lấy kích thước
img_info = []
sample50 = df_train.sample(50, random_state=0)

for fname in tqdm(sample50['image'], desc='Đọc ảnh'):
    path = TRAIN_IMGDIR / fname
    try:
        img = Image.open(path)
        w, h = img.size
        img_info.append({'file': fname, 'width': w, 'height': h, 'mode': img.mode, 'status': 'ok'})
    except Exception as e:
        img_info.append({'file': fname, 'width': None, 'height': None, 'mode': None, 'status': str(e)})

df_img = pd.DataFrame(img_info)
print(df_img['status'].value_counts())
df_img.head(10)

## 5. Phân tích dữ liệu ban đầu (EDA)

In [ ]:
# Biểu đồ 1: Phân bố số sản phẩm trong mỗi label_group
group_sizes = df_train.groupby('label_group').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(group_sizes.values, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Số sản phẩm trong nhóm')
axes[0].set_ylabel('Số nhóm')
axes[0].set_title('Phân bố kích thước nhóm sản phẩm (label_group)')

# Top 20 nhóm lớn nhất
top20 = group_sizes.sort_values(ascending=False).head(20)
axes[1].bar(range(20), top20.values, color='coral')
axes[1].set_xlabel('Top 20 nhóm lớn nhất')
axes[1].set_ylabel('Số sản phẩm')
axes[1].set_title('Top 20 nhóm sản phẩm có nhiều ảnh nhất')

plt.tight_layout()
plt.savefig('../results/bieu_do_1_phan_bo_nhom.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ 1')

In [ ]:
# Biểu đồ 2: Phân bố kích thước ảnh (width & height)
# Lấy mẫu 200 ảnh để kiểm tra nhanh
sample200 = df_train.sample(200, random_state=1)
widths, heights = [], []

for fname in tqdm(sample200['image'], desc='Đọc kích thước ảnh'):
    try:
        img = Image.open(TRAIN_IMGDIR / fname)
        widths.append(img.size[0])
        heights.append(img.size[1])
    except:
        pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(widths, bins=30, color='mediumseagreen', edgecolor='white')
axes[0].axvline(224, color='red', linestyle='--', label='Target resize: 224px')
axes[0].set_xlabel('Chiều rộng (pixels)')
axes[0].set_ylabel('Số ảnh')
axes[0].set_title('Phân bố chiều rộng ảnh')
axes[0].legend()

axes[1].hist(heights, bins=30, color='mediumpurple', edgecolor='white')
axes[1].axvline(224, color='red', linestyle='--', label='Target resize: 224px')
axes[1].set_xlabel('Chiều cao (pixels)')
axes[1].set_ylabel('Số ảnh')
axes[1].set_title('Phân bố chiều cao ảnh')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/bieu_do_2_kich_thuoc_anh.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Đã lưu biểu đồ 2')
print(f'Width  – mean: {np.mean(widths):.0f}px, min: {min(widths)}px, max: {max(widths)}px')
print(f'Height – mean: {np.mean(heights):.0f}px, min: {min(heights)}px, max: {max(heights)}px')

In [ ]:
# Biểu đồ 3: Xem thử một số ảnh trong cùng 1 label_group
# Chọn một nhóm có nhiều ảnh
sample_group = group_sizes[group_sizes >= 3].index[0]
group_imgs = df_train[df_train['label_group'] == sample_group]['image'].tolist()[:6]

fig, axes = plt.subplots(1, len(group_imgs), figsize=(15, 4))
fig.suptitle(f'Các ảnh cùng label_group: {sample_group}', fontsize=13)

for ax, fname in zip(axes, group_imgs):
    try:
        img = Image.open(TRAIN_IMGDIR / fname).convert('RGB')
        img = img.resize((224, 224))
        ax.imshow(img)
    except:
        ax.text(0.5, 0.5, 'Lỗi', ha='center')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../results/bieu_do_3_mau_cung_nhom.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ 3')

## 6. Tiền xử lý ảnh

In [ ]:
import torch
import torchvision.transforms as T

# Pipeline tiền xử lý chuẩn ImageNet
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def load_image(path):
    """Đọc ảnh, chuyển RGB, áp dụng transform."""
    img = Image.open(path).convert('RGB')
    return transform(img).unsqueeze(0)  # [1, 3, 224, 224]

# Thử với 1 ảnh
test_fname = df_train['image'].iloc[0]
tensor = load_image(TRAIN_IMGDIR / test_fname)
print(f'Tensor shape: {tensor.shape}')  # Kỳ vọng: torch.Size([1, 3, 224, 224])
print('✅ Pipeline tiền xử lý hoạt động')

## 7. Baseline – Trích đặc trưng bằng ResNet50 + Cosine Similarity

> **Baseline này dùng tập con nhỏ (500 ảnh) để chạy nhanh.** Tuần 3 sẽ mở rộng toàn bộ dataset + tích hợp FAISS.

In [ ]:
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Sử dụng: {device}')

# Load ResNet50 pre-trained, bỏ lớp FC cuối → lấy vector 2048 chiều
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = torch.nn.Identity()  # Bỏ lớp classification
resnet = resnet.to(device).eval()

print('✅ Load ResNet50 thành công')

In [ ]:
# Trích đặc trưng cho tập con 500 ảnh
SUBSET_SIZE = 500
df_sub = df_train.sample(SUBSET_SIZE, random_state=42).reset_index(drop=True)

features = []
valid_idx = []

with torch.no_grad():
    for i, row in tqdm(df_sub.iterrows(), total=len(df_sub), desc='Trích đặc trưng'):
        try:
            tensor = load_image(TRAIN_IMGDIR / row['image']).to(device)
            feat = resnet(tensor).squeeze().cpu().numpy()  # (2048,)
            features.append(feat)
            valid_idx.append(i)
        except Exception as e:
            pass  # Bỏ qua ảnh lỗi

features = np.array(features)  # (N, 2048)
df_valid = df_sub.loc[valid_idx].reset_index(drop=True)

print(f'\nSố ảnh trích được đặc trưng: {len(features)}')
print(f'Shape vector đặc trưng    : {features.shape}')

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# L2-normalize để dùng cosine similarity = inner product
features_norm = normalize(features, norm='l2')

# Tính cosine similarity giữa tất cả các cặp
sim_matrix = cosine_similarity(features_norm)  # (N, N)
np.fill_diagonal(sim_matrix, -1)  # Loại bỏ self-similarity

print(f'Ma trận similarity shape: {sim_matrix.shape}')
print('✅ Tính similarity thành công')

## 8. Đánh giá Baseline – Precision@K và Recall@K

In [ ]:
def precision_at_k(sim_matrix, labels, k=5):
    """Tính Precision@K trung bình trên toàn tập."""
    n = len(labels)
    precisions = []
    for i in range(n):
        top_k_idx = np.argsort(sim_matrix[i])[::-1][:k]
        true_label = labels[i]
        hits = sum(labels[j] == true_label for j in top_k_idx)
        precisions.append(hits / k)
    return np.mean(precisions)

def recall_at_k(sim_matrix, labels, k=5):
    """Tính Recall@K trung bình trên toàn tập."""
    n = len(labels)
    recalls = []
    for i in range(n):
        top_k_idx = np.argsort(sim_matrix[i])[::-1][:k]
        true_label = labels[i]
        # Tổng số ảnh cùng nhóm (trừ chính nó)
        total_relevant = sum(1 for j in range(n) if labels[j] == true_label and j != i)
        if total_relevant == 0:
            continue
        hits = sum(labels[j] == true_label for j in top_k_idx)
        recalls.append(hits / total_relevant)
    return np.mean(recalls)

labels = df_valid['label_group'].values

results = []
for k in [1, 3, 5, 10]:
    p = precision_at_k(sim_matrix, labels, k)
    r = recall_at_k(sim_matrix, labels, k)
    results.append({'K': k, 'Precision@K': round(p, 4), 'Recall@K': round(r, 4)})

df_results = pd.DataFrame(results)
print('=== KẾT QUẢ BASELINE (ResNet50 + Cosine Similarity, 500 ảnh) ===')
print(df_results.to_string(index=False))
df_results.to_csv('../results/baseline_metrics.csv', index=False)
print('\n✅ Đã lưu kết quả vào results/baseline_metrics.csv')

In [ ]:
# Biểu đồ kết quả baseline
fig, ax = plt.subplots(figsize=(8, 5))
x = df_results['K']
ax.plot(x, df_results['Precision@K'], marker='o', label='Precision@K', color='steelblue')
ax.plot(x, df_results['Recall@K'],    marker='s', label='Recall@K',    color='coral')
ax.set_xlabel('K')
ax.set_ylabel('Score')
ax.set_title('Baseline – ResNet50 + Cosine Similarity\n(500 ảnh mẫu)')
ax.legend()
ax.set_xticks([1, 3, 5, 10])
plt.tight_layout()
plt.savefig('../results/bieu_do_4_baseline_metric.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ baseline')

## 9. Nhận xét kết quả ban đầu

*(Bổ sung nhận xét của nhóm vào đây – yêu cầu tối thiểu 5 nhận xét)*

1. **Mất cân bằng nhóm:** Phần lớn `label_group` chỉ có 2–3 ảnh, trong khi một số nhóm có đến hàng chục ảnh. Điều này khiến Recall@K bị ảnh hưởng vì hệ thống dễ tìm đúng với nhóm nhỏ hơn nhóm lớn.

2. **ResNet50 pre-trained chưa được fine-tune cho Shopee:** Mô hình được huấn luyện trên ImageNet (phân loại 1000 lớp vật thể tự nhiên), không chuyên biệt cho ảnh sản phẩm thương mại điện tử. Kết quả baseline phản ánh giới hạn này.

3. **Kích thước ảnh rất đa dạng:** Ảnh trong dataset có kích thước từ vài chục đến vài nghìn pixel. Việc resize cứng về 224×224 có thể làm mất chi tiết quan trọng của sản phẩm (họa tiết nhỏ, chữ trên nhãn).

4. **Precision@K giảm khi K tăng:** Đây là hành vi bình thường — càng lấy nhiều kết quả thì tỷ lệ sai càng cao. Trong thực tế, K=5 là mức hợp lý để hiển thị cho người dùng.

5. **Baseline chạy trên 500 ảnh chưa đủ đại diện:** Với 34.000+ ảnh trong dataset thật, kết quả trên tập con nhỏ có thể lạc quan hoặc bi quan hơn thực tế. Tuần 3 cần mở rộng toàn bộ dataset.

6. **Chưa xử lý nhiễu ảnh:** Nhiều ảnh Shopee có watermark, khung viền màu, chữ quảng cáo — đây là nguồn nhiễu lớn ảnh hưởng đến vector đặc trưng mà baseline hiện tại chưa giải quyết.

## 10. Kế hoạch tuần 3

- Mở rộng trích đặc trưng toàn bộ ~34.000 ảnh.
- Tích hợp **FAISS** (IVF index) để thay thế brute-force cosine similarity → tăng tốc tìm kiếm.
- So sánh **ResNet50 vs VGG16** trên cùng tập dữ liệu.
- Đánh giá ảnh hưởng của tiền xử lý (có/không augmentation) lên mAP.